<div style="text-align: center;">
    <h1 style="font-size: 64px; 
               font-weight: 800;
               color: #1f0d4f;
               letter-spacing: 2px;
               margin: 40px 0;">
        LISTA 5
    </h1>
</div>

In [1]:
class HashTable:
    def __init__(self, size, hash_function, method="chaining", second_hash_function=None):
        """
        size: rozmiar tablicy
        hash_function: funkcja przyjmująca klucz i zwracająca liczbę (indeks)
        """
        self.size = size
        self.hash_function = hash_function
        self.second_hash_function = second_hash_function
        self.method = method
        self.count = 0

        if self.method == 'chaining':
            self.table = [[] for _ in range(size)]
        elif self.method =='linear':
            self.table = [None] * size
        elif self.method == 'double':
            self.table = [None] * size
        else:
            raise ValueError("Nieznana metoda")

    def insert(self, key):
        """
        Oblicza indeks za pomocą przekazanej funkcji i wstawia element.
        """
        start_index = self.hash_function(key) % self.size
        
        if self.method == 'chaining':
            self._insert_chaining(start_index, key)
        elif self.method == 'linear':
            self._insert_linear(start_index, key)
        elif self.method == 'double':
            self._insert_double(start_index, key)

    def _insert_chaining(self, index, key):
        self.table[index].append(key)

    def _insert_double(self, idx, key):
            if self.table[idx] is None:
                self.table[idx] = key
                self.count += 1
                return

            step = self.second_hash_function(key)
            
            original_idx = idx
            i = 1
            
            while True:
                new_idx = (idx + step) % self.size
                
                if self.table[new_idx] is None:
                    self.table[new_idx] = key
                    self.count += 1
                    return
                
                idx = new_idx
                i += 1
                
                if idx == original_idx or i > self.size:
                    return

    def _insert_linear(self, index, key):
        if self.count >= self.size:
            print(f"Błąd: Tablica pełna! Nie można wstawić {key}.")
            return

        current_index = index
        
        while self.table[current_index] is not None:
            current_index = (current_index + 1) % self.size
            
            if current_index == index:
                print("Błąd: Przeszliśmy całą tablicę i nie ma miejsca.")
                return

        self.table[current_index] = key
        self.count += 1

    def display(self):
        for i, chain in enumerate(self.table):
            if chain:
                print(f"Indeks {i}: {chain}")
            else:
                print(f"Indeks {i}: (puste)")

# 1. Jak będzie efekt wstawienia kluczu $ 12,44,13,88,23,94,11,39,20,16,5$ do $11$-elementowej tablicy asocjacyjnej, jeżeli funkcja mieszająca ma postać: 
$$ h(i) = (3i+5) \text{mod} 11 $$
a kolizje rozwiązania są metodą łańcuchową?

In [2]:
keys = [12,44,13,88,23,94,11,39,20,16,5]
hash_table = HashTable(len(keys), lambda i: (3*i+5) % 11, 'chaining')

for k in keys:
    hash_table.insert(k)

hash_table.display()

Indeks 0: [13]
Indeks 1: [94, 39]
Indeks 2: (puste)
Indeks 3: (puste)
Indeks 4: (puste)
Indeks 5: [44, 88, 11]
Indeks 6: (puste)
Indeks 7: (puste)
Indeks 8: [12, 23]
Indeks 9: [16, 5]
Indeks 10: [20]


# 2. Jaki będzie wynik poprzedniego zadania, jeżeli do rozwiązywania użyjemy próbkowania liniowego?

In [3]:
keys = [12,44,13,88,23,94,11,39,20,16,5]
hash_table = HashTable(len(keys), lambda i: (3*i+5) % 11, 'linear')

for k in keys:
    hash_table.insert(k)

hash_table.display()

Indeks 0: 13
Indeks 1: 94
Indeks 2: 39
Indeks 3: 16
Indeks 4: 5
Indeks 5: 44
Indeks 6: 88
Indeks 7: 11
Indeks 8: 12
Indeks 9: 23
Indeks 10: 20


# 3. Jaki będzie wynik pierwszego zadania jeśli kolizje usuwane są przy pomocy drugiej funkcji mieszającej

$$ h(k) = 7 - (k \text{mod} 7) \text{?}$$


In [4]:
keys = [12, 44, 13, 88, 23, 94, 11, 39, 20, 16, 5]

# Definicja funkcji
h1 = lambda k: (3*k + 5) % len(keys)
h2 = lambda k: 7 - (k % 7)

hash_table = HashTable(len(keys), h1, 'double', h2)

for k in keys:
    hash_table.insert(k)

hash_table.display()

Indeks 0: 13
Indeks 1: 94
Indeks 2: 23
Indeks 3: 88
Indeks 4: 39
Indeks 5: 44
Indeks 6: 11
Indeks 7: 5
Indeks 8: 12
Indeks 9: 16
Indeks 10: 20


# 6. Zaimplemetuj algorytm sortowania szybkiego w wersji bez rekurencji
oraz działającego w miejscu (ang. in-place), tzn. potrzebującego do wykonania stałego rozmiaru pamięci komputera, niezależnie od wielkości danych wejściowych.

In [5]:
def partition(arr, low, high):
    """
    dzieli tablicę na dwie części względem piwota.
    Elementy mniejsze od piwota trafiają na lewo, większe na prawo.
    Zwraca indeks, na którym ostatecznie znalazł się piwot.
    """
    # Wybieramy ostatni element jako piwot
    pivot = arr[high]
    i = low - 1  # Indeks mniejszego elementu
    
    for j in range(low, high):
        # Jeśli bieżący element jest mniejszy lub równy piwotowi
        if arr[j] <= pivot:
            i += 1
            # Zamień elementy miejscami
            arr[i], arr[j] = arr[j], arr[i]
            
    arr[i + 1], arr[high] = arr[high], arr[i + 1]
    return i + 1

def quick_sort_iterative(arr):
    """
    Główna funkcja sortująca iteracyjnie.
    """
    n = len(arr)
    if n < 2:
        return arr
        
    stack = [(0, n - 1)]
    
    # Dopóki stos nie jest pusty, mamy fragmenty do posortowania
    while stack:
        low, high = stack.pop()
        
        if low < high:
            pi = partition(arr, low, high)
            
            if pi + 1 < high:
                stack.append((pi + 1, high))
                
            # Dodajemy elementy na lewo od piwota
            if pi - 1 > low:
                stack.append((low, pi - 1))


data = [12, 44, 13, 88, 23, 94, 11, 39, 20, 16, 5]

print(f"Przed sortowaniem: {data}")

quick_sort_iterative(data)

print(f"Po sortowaniu:     {data}")

assert data == sorted([12, 44, 13, 88, 23, 94, 11, 39, 20, 16, 5])

Przed sortowaniem: [12, 44, 13, 88, 23, 94, 11, 39, 20, 16, 5]
Po sortowaniu:     [5, 11, 12, 13, 16, 20, 23, 39, 44, 88, 94]


# 7. Kod zadania 7 znajduje się w pliku animacje.py. Pliki gif: QuickSortVisualization.gif, BubbleSortVisualization.gif